# Notebook 06 — Release preparation

**Purpose.** Reproducibility audit, notebook-output policy, aggregate privacy suppression check, allowlisted export, secret and history scans, and the gated GitHub publication. **GPU:** off. Raw data, per-row predictions, checkpoints and credentials are never exported.

In [1]:
import os, sys, json, subprocess
from pathlib import Path
REPO = Path.cwd().resolve() if (Path.cwd() / "src" / "cape_eeg").exists() else Path.cwd().resolve().parent
sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("CAPE_ROOT", str(REPO.parent)); os.environ.setdefault("HMS_DATA_ROOT", os.environ["CAPE_ROOT"])
os.environ["PYTHONWARNINGS"] = "ignore"
from cape_eeg.paths import resolve_workspace, redact
from cape_eeg.status import read_json, Ledger
ws = resolve_workspace()
def run(cmd, **kw):
    """Run a repository script as a bounded subprocess; prints filtered output (no secrets, no identifiers)."""
    p = subprocess.run([sys.executable, str(REPO / "scripts" / cmd[0]), *cmd[1:]], capture_output=True, text=True, env=os.environ, **kw)
    for line in (p.stdout + p.stderr).splitlines():
        if line.strip() and not any(w in line for w in ("Warning", "warn", "Found GPU", "Minimum and", "(8.0)")):
            print(line)
    if p.returncode != 0:
        raise RuntimeError(f"{cmd[0]} exited with {p.returncode}")
print("repo:", redact(REPO, ws)); print("workspace root:", redact(ws.root, ws)); print("data root:", redact(ws.data, ws)); print("private:", redact(ws.private, ws))


repo: $CAPE_ROOT/cape-eeg
workspace root: $CAPE_ROOT
data root: $CAPE_ROOT
private: $CAPE_ROOT/private


## Reproducibility audit: hashes and tests

In [2]:
print('split_hash', read_json(ws.manifests / 'split_summary.json')['split_hash']); print('preprocess/cache', {p.parent.name: read_json(p)['cache_hash'] for p in ws.cache.glob('*/manifest.json')})
lock = read_json(ws.manifests / 'protocol_lock.json'); print('protocol_hash', lock['protocol_hash'] if lock else None)
p = subprocess.run([sys.executable, '-m', 'pytest', '-q', str(REPO / 'tests')], capture_output=True, text=True, cwd=REPO); print(p.stdout.strip().splitlines()[-1])

split_hash 4455edaac7cb7e5d
preprocess/cache {'2406ce595fbd7457': 'af515c778666c076'}
protocol_hash 380269a4a6db7334


155 passed in 22.79s


## Export precheck: allowlist, forbidden files, identifier scan, secret scan

In [3]:
run(['release_precheck.py'])

content precheck: PASS files=178 bytes=15435333 export_sha256=6dd8440dba5cf0e5...
history scan: PASS objects=206 hits=0


## Publication state
Push happens only through `scripts/publish.py` with an explicit destination and a passing scan; this notebook records the state, it does not push.

In [4]:
rec = read_json(ws.approvals / 'publication_record.json'); print(json.dumps(rec, indent=1) if rec else 'publication: NOT_RUN')

{
 "owner": "Hisernberg",
 "repository": "cape-eeg",
 "branch": "main",
 "visibility": "public",
 "export_sha256": "6dd8440dba5cf0e55a594cacd1acc20dbd77c7ab2a7b75682350ce74de98a046",
 "content_scan": "PASS",
 "history_scan": "PASS",
 "commit": "57318e9c5293a9806ff33bb0998adb727e44a25d",
 "n_files": 178,
 "total_bytes": 15421061,
 "timestamp": "2026-09-07T14:32:10+00:00",
 "push": "PASS",
 "remote_commit_verified": true,
 "remote_url": "https://github.com/Hisernberg/cape-eeg",
 "push_method": "gh CLI (token from local credential store)",
 "repository_created": true,
 "verified_at": "2026-09-07T14:33:02+00:00",
 "public_visibility_approved": true,
 "weights_release_approved": false,
 "raw_data": false,
 "row_predictions": false
}
